# 08 Connecting Ollama and gpt-oss:120b

## 本地模型接入不是把云 API 换成 localhost，而是重新理解运行边界

到了这一章，整套项目第一次真正碰到运行层现实。前面几章已经把问题讲清楚了：模型不是执行器，Agent 是任务运行时，MCP 是能力层。本地 MCP Server 也已经作为能力面原型被放出来。接下来真正需要解决的问题是，这些结构如何落到一个本地可运行的模型链路上。

表面看，这一步像是一个很简单的接线问题：把模型调用目标从某个云端 API 改成 `Ollama`，然后选择 `gpt-oss:120b` 作为底层模型，剩下的就只是请求格式差异。但如果只把它理解成“换个 endpoint”，会直接低估本地运行带来的结构变化。

因为一旦模型开始在本地运行，很多在托管平台里被默认吞掉的问题都会重新出现：延迟、吞吐、上下文长度压力、结构化输出稳定性、工具调用服从度、失败重试成本、资源竞争、调试方式。这些都不再是平台替你兜底的背景噪音，而会变成 Agent Runtime 设计的一部分。

所以，这一章真正要回答的不是“怎么连上 Ollama”，而是：**为什么本地模型接入会改变你对整个 Agent 系统的工程判断。**

## 先给结论

这一章最重要的判断可以压缩成一句话：

> 把 `Ollama + gpt-oss:120b` 接进 Agent 系统，真正变化的不是模型供应商，而是运行时开始必须显式承担原本被平台隐藏的稳定性、结构约束和资源成本问题。

这句话里有几个重点：

- 本地模型接入不是“同样的系统换个模型名”
- 模型能力边界会更直接暴露在 Runtime 设计里
- 结构化输出和 tool use 的稳定性会从“模型功能”变成“系统联合结果”
- 本地运行更适合做体系理解，因为很多真问题不会再被平台抽象层遮住

也正因为如此，这一章的重点不会是安装教程，而是运行链理解。

## 1. 为什么这套项目适合用本地模型来承载

如果目标只是快速做一个演示，直接接云端模型通常更省心。但这套项目本来就不是单纯做“能跑的 demo”，而是要把系统机制讲透。在这种目标下，本地模型反而更合适。

原因很直接：

- 它更能暴露真实的运行边界
- 它能让 Agent Runtime 的调度逻辑和模型表现被直接观察
- 它避免把系统能力误解成平台能力
- 它更符合“我理解整个栈，而不是只会调托管 API”的展示方向

这并不是说本地一定优于云端，而是说对这个项目的叙事来说，本地更有解释力。因为一旦模型在你自己的机器上运行，很多关于结构化输出稳定性、上下文长度、动作意图服从度的判断，就必须靠你自己系统化处理，而不能假装它们不存在。

## 2. 为什么是 Ollama：它的价值不是“简单”，而是把本地模型接入变成稳定接口

选择 `Ollama` 的原因不应该被写成“因为好装”这种表层理由。更准确的说法是：它把本地模型运行从“直接操作底层推理栈”抽象成了一个比较稳定的本地服务接口。

这会带来几个实际好处：

- Agent Runtime 可以把模型调用看成稳定服务，而不是设备级脚本
- Notebook、后续脚本和本地 runtime 可以共用一致调用方式
- 模型切换、参数调整、上下文实验可以被放在统一调用面下进行

也就是说，`Ollama` 在这套项目中的价值，不是简化安装体验，而是提供一个足够稳定的本地模型宿主，让前面讨论的运行时结构可以真正落下来。

## 3. 为什么选 gpt-oss:120b：不是为了参数数字，而是为了任务型行为质量

模型选择如果只写成“用了一个更大的模型”，会显得很空。更像样的表达应该是：之所以选 `gpt-oss:120b`，是因为这套项目要验证的不是闲聊质量，而是任务型行为，包括：

- 指令服从度
- 结构化输出稳定性
- 在多轮上下文里维持角色和约束的能力
- 对工具调用与资源读取的倾向是否可被塑形

换句话说，这里看重的不是模型能不能写漂亮句子，而是它能不能在一个被约束的任务环境里，稳定地产生可被 Runtime 消费的结构化输出。

这也是为什么这一章不该把模型选择写成品牌偏好。真正重要的是：**这个模型是否足以承接前面几章构建出来的任务结构。**

## 4. 本地调用链真正长什么样

一旦模型进入本地运行链，系统结构可以被更清楚地拆出来：

- Notebook 或 Runtime 负责拼装消息与控制信息
- 本地调用层负责把请求发给 `Ollama`
- `Ollama` 负责把请求交给 `gpt-oss:120b`
- 模型返回自然语言或结构化动作意图
- Runtime 再决定下一步是回答、读 resource、调 tool，还是继续循环

这条链路看似普通，但有一个关键变化：模型不再被放在一个“大而全的产品 API”后面，而是被明确放在你自己的运行时图里。这样一来，哪些是模型负责的，哪些是 Runtime 负责的，哪些是能力层负责的，都会更容易被看清。

In [ ]:
ollama_request = {
    "model": "gpt-oss:120b",
    "messages": [
        {
            "role": "system",
            "content": "你是一个任务型 Agent 的推理核心。信息不足时优先选择读取资源或调用工具，不要凭空补全关键事实。",
        },
        {
            "role": "user",
            "content": "请根据岗位 JD 和候选人资料，输出能力匹配分析。",
        },
    ],
    "format": "json",
}

print(ollama_request)

## 5. 本地模型最先暴露的问题，通常不是“能不能回答”，而是“能不能稳定按结构回答”

在很多云端体验里，人们很容易默认结构化输出是理所当然的：给 schema，模型就大致会按 schema 来；给工具列表，模型就大致会往工具路径走。但到了本地模型，这种默认值往往会被打破，或者至少被削弱。

于是你会发现，真正要验证的不是模型能不能写一段像样的话，而是这些更系统的问题：

- 它能不能稳定保持 JSON 结构
- 它能不能在 system prompt 约束下优先选择工具路径
- 它能不能在长一点的上下文里维持角色一致性
- 它拿到 tool result 之后会不会真的改变后续输出

这也是为什么本地接入不应被写成“换个 endpoint”这种轻描淡写的动作。因为本地模型会迫使你面对系统的真实结构压力。

## 6. 本地模型让 Runtime 的职责变得更重，而不是更轻

很多人会下意识觉得，本地模型更可控，所以 Runtime 压力反而会小一点。实际情况常常相反。

因为一旦没有强平台兜底，Runtime 必须更主动地承担以下工作：

- 重新强调 system prompt 和输出边界
- 对结构化结果做更严格的校验
- 在失败时重试、澄清或压缩上下文
- 控制单轮上下文大小，避免不必要的推理负担
- 决定什么时候该把“模型不稳定”转化成“系统退路”

换句话说，本地模型不会自动让 Agent 更强，它只会让你更清楚看到 Agent Runtime 到底应该承担什么。

## 7. 延迟和成本在本地环境里会重新变成设计变量

云端平台当然也有延迟和成本，但在很多使用体验里，它们被包进了“调用一次模型服务”的抽象里。到了本地运行，这两个变量会以更原始的方式进入设计判断。

比如：

- 一次长上下文请求会显著拖慢整个 Agent loop
- 多次无意义的工具选择试探会直接放大交互等待
- 冗余的历史上下文回填会立刻转化成推理负担

于是，很多原本看起来只是“写法差异”的东西，会直接变成运行时效率问题。比如为什么要压缩 tool result，为什么要裁剪历史消息，为什么要有终止条件，为什么不该让 Agent 无节制地继续想。这些在本地环境里都会显得格外具体。

## 8. 接入本地模型之后，真正该验证什么

如果这一步只验证“请求发过去能收到响应”，那几乎什么都没验证。更有价值的验证项应该围绕 Agent 系统真正关心的行为展开：

- 指令跟随是否足够稳定
- system prompt 对工具优先级的塑形是否明显
- 在存在 resource 和 tool 的情况下，模型是否会区分不同动作路径
- 结构化输出能否被 Runtime 可靠解析
- tool result 回填后，后续回答是否发生合理变化

这些验证项之所以重要，是因为它们决定了本地模型到底适不适合作为后续 Runtime 的推理核心。模型不需要完美，但至少要能在这些关键环节上被系统稳定约束。

In [ ]:
validation_matrix = [
    "instruction adherence",
    "json or structured output stability",
    "tool-use inclination under system constraints",
    "response change after tool/resource feedback",
    "latency under multi-step loop",
]

for item in validation_matrix:
    print(item)

## 9. 本地运行的另一个价值：可观测性更真实

本地模型接入对这套项目还有一个很重要的好处，就是你能更真实地看到系统在做什么。

在很多托管场景下，模型调用过程对开发者来说像一个黑箱。你拿到最终输出，最多再拿到一点 usage 信息。而本地环境更容易让你把以下东西都放进观察范围：

- 每一轮发给模型的实际 messages
- 当前 Runtime 是否把资源和工具信息正确拼进了上下文
- 模型在什么地方开始偏离结构要求
- 哪些延迟来自模型，哪些来自 Runtime 决策或能力调用

这对于一个强调“系统理解”的项目尤其重要。因为你要展示的不是“我能从黑箱里拿到答案”，而是“我知道系统是怎么一步步运行的”。

## 10. 本地模型的局限不该被掩饰，而该被纳入系统设计

如果这套项目是给人看技术判断力，那么对本地模型的局限保持诚实，反而比一味美化更有说服力。

例如，完全可以明确承认：

- 本地模型在某些结构化输出场景下可能不如强托管模型稳定
- 在多步 Agent 循环里，延迟和上下文压力会更快暴露
- 工具调用服从度可能需要更强的 Runtime 约束来补足

但这不是缺点展示，而是系统成熟度展示。因为一个真正做系统的人，重点从来不是证明模型永远完美，而是证明系统知道如何与模型的真实边界合作。

## 11. 为什么本地模型接好之后，下一步一定是 Runtime

走到这里，其实已经很清楚了：模型接入本身并不会自动生成 Agent。它只是把一个推理核心放进了本地运行链。真正决定系统是否像 Agent 的，还是后面的 Runtime：

- 如何维护目标和状态
- 如何决定读 resource 还是调 tool
- 如何把模型输出解释成动作
- 如何把结果写回并继续循环
- 如何在本地模型不稳定时做出恢复决策

也就是说，`Ollama + gpt-oss:120b` 解决的是“谁来推理”，而 `Agent Runtime` 解决的是“推理如何变成任务推进”。这两层必须连起来看，前者才不会退化成单纯的本地聊天接口。

## 12. 本章结论

这一章最值得保留的判断有这些：

- 本地模型接入不是换 endpoint，而是重新面对运行边界。
- `Ollama` 的价值在于把本地模型变成稳定宿主接口，而不只是安装方便。
- 选择 `gpt-oss:120b` 的重点不是参数数字，而是任务型行为质量。
- 本地环境会更直接暴露结构化输出、工具服从度、上下文压力和延迟问题。
- 模型接入解决的是推理核心问题，真正让系统像 Agent 的仍然是后续 Runtime。

下一章会进入这套系统最核心的实现层：Agent Runtime。前面所有关于 Prompt、Role、Tool Calling、MCP、Local Model 的讨论，都会在那里汇成一个真正的任务闭环。